# College Station, TX LVT Shift Notebook

This notebook models a Land Value Tax shift for College Station, Texas, using data from
the Brazos Central Appraisal District (Brazos CAD).
Reference spec: `examples/skills/college_station/FETCH.md`.

**Data sources:**
- Geometry: 2025 Certified Parcel Shapefile (Brazos CAD)
- Valuation: 2025 Certified PACS Export — `APPRAISAL_INFO.TXT` (fixed-width format)
- Zoning: City of College Station open-data FeatureServer

**Policy defaults:**
- Scope: parcels where `SITUS_CITY = 'COLLEGE STATION'`; real property only
- TAMU/state-owned parcels (`ex_exempt = 'T'`) shown as "Exempt / Government" and excluded from LVT modeling
- Reform: split-rate LVT scenarios (2:1, 4:1, 8:1)
- Current tax: sum of CS, BC, and CSISD levies with entity-specific homestead exemptions applied


In [1]:
import io
import sys
import zipfile
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import seaborn as sns
from shapely.geometry import Point, Polygon

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "lvt_utils.py").exists():
    REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

from lvt_utils import (
    calculate_category_tax_summary,
    model_split_rate_tax,
    print_category_tax_summary,
)
from policy_analysis import (
    analyze_land_by_improvement_share,
    analyze_parking_lots,
    analyze_vacant_land,
    print_parking_analysis_summary,
    print_vacant_land_summary,
)
from census_utils import get_census_data_with_boundaries, match_to_census_blockgroups
from viz import create_quintile_summary

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 120)


In [2]:
data_dir = REPO_ROOT / "examples" / "data" / "college_station"
data_dir.mkdir(parents=True, exist_ok=True)

# ── Data source URLs ──────────────────────────────────────────────────────────
SHP_URL_2025   = "https://brazoscad.org/wp-content/uploads/2025/11/Public_Parcel_Boundary_certified.zip"
PACS_URL_2025  = "https://brazoscad.org/wp-content/uploads/2025/08/2025-CERTIFIED-EXPORT.zip"
ZONING_URL     = "https://gis.cstx.gov/csgis/rest/services/_OpenData/OpenData_PDS/MapServer/17/query"

CITY_FILTER_VALUE  = "COLLEGE STATION"
BRAZOS_COUNTY_FIPS = "48041"

# ── 2025 Adopted tax rates (per $100 of assessed value) ──────────────────────
# Source: https://brazoscad.org/tax-information/adopted-tax-rates/
MILLAGE_CS    = 0.511872   # City of College Station
MILLAGE_BC    = 0.419700   # Brazos County
MILLAGE_CSISD = 0.975300   # College Station ISD
# ESDs (0.02–0.09 per $100) and MUDs (~$1.00) apply only to parcels within
# those geographic sub-districts. Incorporate via APPRAISAL_ENTITY_INFO.TXT
# if full-stack accuracy is needed.

# ── Entity-specific exemption amounts (2025) ─────────────────────────────────
# City of College Station
CS_HOMESTEAD_RATE = 0.20     # 20% of appraised value
CS_OV65_AMT       = 30_000  # flat dollar, over-65

# Brazos County — verify optional exemptions at brazoscountytx.gov
BC_HOMESTEAD_AMT  = 0        # TODO: update if BC offers optional homestead
BC_OV65_AMT       = 0        # TODO: update if BC offers over-65 flat

# College Station ISD — mandatory per Tex. Ed. Code §11.13(b)
CSISD_HOMESTEAD_AMT = 40_000
CSISD_OV65_AMT      = 10_000  # additional local over-65 exemption

# ── PACS APPRAISAL_INFO.TXT fixed-width byte offsets (layout v8.0.25) ────────
# All offsets are 0-based Python slice indices (start inclusive, end exclusive).
# Source: FETCH.md (which corrects the 1-based PDF positions to 0-based).
PROP_COLS = {
    "prop_id":            (0,    12),
    "prop_type_cd":       (12,   17),
    "geo_id":             (546,  596),
    "py_owner_name":      (608,  678),
    "situs_num":          (4459, 4474),
    "situs_street":       (1049, 1099),
    "situs_city":         (1109, 1139),
    "situs_zip":          (1139, 1149),
    "appraised_val":      (1915, 1930),
    "ten_percent_cap":    (1930, 1945),
    "assessed_val":       (1945, 1960),
    "land_hstd_val":      (1795, 1810),
    "land_non_hstd_val":  (1810, 1825),
    "imprv_hstd_val":     (1825, 1840),
    "imprv_non_hstd_val": (1840, 1855),
    "land_acres":         (2771, 2791),
    "hs_exempt":          (2608, 2609),
    "ov65_exempt":        (2609, 2610),
    "ex_exempt":          (2670, 2671),
    "land_state_cd":      (2741, 2751),
    "imprv_state_cd":     (2731, 2741),
}

# ── Cache file paths ──────────────────────────────────────────────────────────
geometry_cache  = data_dir / "cs_geometry_2025cert.parquet"
appraisal_cache = data_dir / "cs_appraisal_info_2025cert.parquet"
zoning_cache    = data_dir / "cs_zoning_2025.parquet"
osm_cache       = data_dir / "cs_osm_surface_parking_2025.parquet"
mapping_cache   = data_dir / "cs_mapping_ready_2025cert.parquet"


In [3]:
def parse_fixed_width(file_obj, col_specs, encoding="latin-1"):
    # Parse a PACS fixed-width text file. file_obj may be a path or open binary stream.
    if hasattr(file_obj, "read"):
        lines = io.TextIOWrapper(file_obj, encoding=encoding).readlines()
    else:
        with open(file_obj, "r", encoding=encoding) as fh:
            lines = fh.readlines()
    rows = [
        {k: ln[s:e].strip() for k, (s, e) in col_specs.items()}
        for ln in lines
        if ln.strip()
    ]
    return pd.DataFrame(rows)


def _pick_parcel_shp(shp_files):
    """Pick the parcel boundary .shp from the zip."""
    annotation_keywords = {
        "ABSTRACT", "SUBDIV", "ADDRESS", "ID", "XREF", "HOOK",
        "GRID", "TEXT", "NUMBER", "NAME", "BOUNDARY", "LINE",
        "CREEK", "LAKE", "POND", "ROAD", "RAILROAD", "UDI",
        "PHASE", "BLOCK", "LOT", "CENTER",
    }

    def score(path):
        fname = path.split("/")[-1].upper().replace(".SHP", "")
        parts = set(fname.split("_"))
        if fname in ("PUBLIC_PARCEL_BOUNDARY_CERTIFIED", "PUBLIC_PARCEL_BOUNDARY",
                     "PARCEL", "PARCELS"):
            return 10
        if "PARCEL" in fname and not (parts & annotation_keywords):
            return 5
        if "PARCEL" in fname:
            return 2
        if not (parts & annotation_keywords):
            return 1
        return 0

    ranked = sorted(shp_files, key=score, reverse=True)
    print(f"  Shapefile scores: {[(s.split('/')[-1], score(s)) for s in ranked[:5]]}")
    return ranked[0]


def load_shapefile():
    if geometry_cache.exists():
        print(f"Loading geometry cache: {geometry_cache.name}")
        return gpd.read_parquet(geometry_cache)

    print("Downloading 2025 certified shapefile from Brazos CAD...")
    resp = requests.get(SHP_URL_2025, stream=True, timeout=600)
    resp.raise_for_status()
    buf = io.BytesIO()
    for chunk in resp.iter_content(256 * 1024):
        buf.write(chunk)
    buf.seek(0)
    print(f"  Downloaded {buf.tell() / 1e6:.1f} MB")

    import tempfile
    with zipfile.ZipFile(buf) as zf:
        shp_files = [n for n in zf.namelist() if n.lower().endswith(".shp")]
        if not shp_files:
            raise FileNotFoundError(f"No .shp in ZIP. Contents: {zf.namelist()}")
        chosen_shp = _pick_parcel_shp(shp_files)
        print(f"  Using: {chosen_shp}")
        with tempfile.TemporaryDirectory() as tmpdir:
            zf.extractall(tmpdir)
            gdf_all = gpd.read_file(Path(tmpdir) / chosen_shp)

    print(f"  Shapefile columns: {gdf_all.columns.tolist()}")
    print(f"  Total rows: {len(gdf_all):,}")

    # Filter to College Station — prefer addr_city, fall back to any CITY column
    for city_candidate in ("addr_city", "situs_city", "SITUS_CITY", "ADDR_CITY"):
        if city_candidate in gdf_all.columns:
            gdf_cs = gdf_all[
                gdf_all[city_candidate].str.upper().str.strip() == CITY_FILTER_VALUE
            ].copy()
            print(f"  Filtered on '{city_candidate}': {len(gdf_cs):,} College Station parcels")
            break
    else:
        # Last resort: any column with CITY in the name
        city_col = next((c for c in gdf_all.columns if "CITY" in c.upper()), None)
        if city_col:
            gdf_cs = gdf_all[
                gdf_all[city_col].str.upper().str.strip() == CITY_FILTER_VALUE
            ].copy()
            print(f"  Filtered on '{city_col}': {len(gdf_cs):,} College Station parcels")
        else:
            gdf_cs = gdf_all.copy()
            print("Warning: no city column found; keeping all parcels.")

    # Use geo_id (lowercase) as the PACS join key — it's the geographic account ID.
    # PROP_ID is the internal numeric ID and does NOT match PACS geo_id.
    if "geo_id" in gdf_cs.columns:
        gdf_cs = gdf_cs.rename(columns={"geo_id": "GEO_ID"})
        print("  Join key: 'geo_id' → 'GEO_ID'")
    else:
        # Fallback: look for any geo/account ID column, excluding PROP_ID
        geo_col = next(
            (c for c in gdf_cs.columns
             if c.upper().replace("_", "") in ("GEOID", "GEONUM", "ACCOUNTNO", "PARCELID")),
            None,
        )
        if geo_col:
            gdf_cs = gdf_cs.rename(columns={geo_col: "GEO_ID"})
            print(f"  Join key: '{geo_col}' → 'GEO_ID' (fallback)")
        else:
            print("Warning: no geo_id join key found.")

    if not gdf_cs.crs:
        gdf_cs = gdf_cs.set_crs("EPSG:4326")
    elif gdf_cs.crs.to_epsg() != 4326:
        gdf_cs = gdf_cs.to_crs("EPSG:4326")

    gdf_cs.to_parquet(geometry_cache, index=False)
    print(f"Saved: {geometry_cache.name}  ({len(gdf_cs):,} parcels)")
    return gdf_cs


def load_pacs_appraisal_info(diagnose=False):
    """Load PACS APPRAISAL_INFO.TXT — all Brazos County records (no city filter).

    situs_city is blank for most records in the county export; CS filtering is done
    at merge time using the geo_id join key against the shapefile.
    """
    if appraisal_cache.exists() and not diagnose:
        print(f"Loading PACS appraisal info cache: {appraisal_cache.name}")
        return pd.read_parquet(appraisal_cache)

    pacs_zip_path = data_dir / "_pacs_2025cert_raw.zip"
    if not pacs_zip_path.exists():
        print("Downloading 2025 PACS export (large file, may take several minutes)...")
        with requests.get(PACS_URL_2025, stream=True, timeout=3600) as resp:
            resp.raise_for_status()
            total = int(resp.headers.get("content-length", 0))
            downloaded = 0
            with open(pacs_zip_path, "wb") as f:
                for chunk in resp.iter_content(1024 * 1024):
                    f.write(chunk)
                    downloaded += len(chunk)
                    if total and downloaded % (50 * 1024 * 1024) < 1024 * 1024:
                        print(f"  {downloaded / 1e6:.0f} / {total / 1e6:.0f} MB")
        print(f"  Download complete: {downloaded / 1e6:.0f} MB")

    with zipfile.ZipFile(pacs_zip_path) as zf:
        names = zf.namelist()
        info_file = next(
            (
                n for n in names
                if "APPRAISAL_INFO" in n.upper()
                and "ENTITY" not in n.upper()
                and "IMPROVEMENT" not in n.upper()
                and "LAND" not in n.upper()
                and "STATE" not in n.upper()
                and "ABSTRACT" not in n.upper()
            ),
            None,
        )
        if info_file is None:
            raise FileNotFoundError(
                "APPRAISAL_INFO.TXT not found in ZIP.\nContents:\n" + "\n".join(names)
            )
        print(f"  Parsing: {info_file}")

        if diagnose:
            with zf.open(info_file) as f:
                raw_line = io.TextIOWrapper(f, encoding="latin-1").readline()
            print(f"  Line length: {len(raw_line)} chars")
            print("  Key field extracts from first row:")
            for field, (s, e) in PROP_COLS.items():
                val = raw_line[s:e].strip() if e <= len(raw_line) else "<out of range>"
                print(f"    [{s}:{e}]  {field:<22} = {repr(val)}")
            return None

        with zf.open(info_file) as f:
            df_all = parse_fixed_width(f, PROP_COLS)

    # Keep real property only; city filtering happens at merge time via geo_id
    df_real = df_all[df_all["prop_type_cd"].str.strip().isin(["R", "M", ""])].copy()
    print(f"  Total rows parsed: {len(df_all):,}  |  Real/mobile property: {len(df_real):,}")

    df_real.to_parquet(appraisal_cache, index=False)
    print(f"Saved: {appraisal_cache.name}  (all Brazos County real-property records)")
    return df_real


def fetch_arcgis_geojson(query_url, where="1=1", chunk_size=2000, headers=None):
    session = requests.Session()
    count_resp = session.get(
        query_url,
        params={"f": "json", "where": where, "returnCountOnly": "true"},
        timeout=60,
        headers=headers,
    )
    count_resp.raise_for_status()
    total = int(count_resp.json().get("count", 0))
    rows = []
    for offset in range(0, total, chunk_size):
        params = {
            "f": "geojson",
            "where": where,
            "outFields": "*",
            "resultOffset": offset,
            "resultRecordCount": chunk_size,
            "outSR": 4326,
        }
        resp = session.get(query_url, params=params, timeout=180, headers=headers)
        resp.raise_for_status()
        feats = resp.json().get("features", [])
        if not feats:
            break
        rows.extend(feats)
        print(f"  {min(offset + len(feats), total):,} / {total:,}")
    if not rows:
        return gpd.GeoDataFrame(columns=["geometry"], geometry="geometry", crs="EPSG:4326")
    return gpd.GeoDataFrame.from_features(rows, crs="EPSG:4326")


def load_zoning_layer():
        _cached_z = gpd.read_parquet(zoning_cache)

        if len(_cached_z) > 0:
            print(f"Loading zoning cache: {zoning_cache.name}")
            return _cached_z
        print("Stale empty zoning cache; re-fetching...")

    headers = {
        "User-Agent": "Mozilla/5.0 (compatible; LVTShift/1.0)",
        "Referer": "https://maps.cstx.gov/",
    }
    try:
        zoning_gdf = fetch_arcgis_geojson(ZONING_URL, where="1=1", chunk_size=2000, headers=headers)
    except Exception as exc:
        print(f"Primary zoning endpoint failed ({exc}); trying open-data portal.")
        # Try ArcGIS Hub GeoJSON download
        for fallback in [
            "https://opendata.arcgis.com/datasets/7b5a1e93abde4ce9a3e92e32af536ee4_0.geojson",
            "https://data-cstx.opendata.arcgis.com/api/download/v1/items/zoning/geojson",
        ]:
            try:
                resp = requests.get(fallback, timeout=120)
                resp.raise_for_status()
                zoning_gdf = gpd.GeoDataFrame.from_features(
                    resp.json()["features"], crs="EPSG:4326"
                )
                break
            except Exception:
                continue
        else:
            print("Warning: all zoning endpoints failed; ZONING_CLASS will be null.")
            return gpd.GeoDataFrame(columns=["geometry"], geometry="geometry", crs="EPSG:4326")

    if len(zoning_gdf) == 0:
        print("Warning: zoning returned zero records.")
        return zoning_gdf

    zoning_gdf.to_parquet(zoning_cache, index=False)
    print(f"Saved: {zoning_cache.name}  ({len(zoning_gdf):,} zones)")
    return zoning_gdf


def fetch_osm_surface_parking(bounds):
    minx, miny, maxx, maxy = bounds
    overpass_eps = [
        "https://overpass-api.de/api/interpreter",
        "https://lz4.overpass-api.de/api/interpreter",
    ]

    def fetch_tile(tx_minx, tx_miny, tx_maxx, tx_maxy):
        query = (
            f"[out:json][timeout:120];"
            f"(way[\"amenity\"=\"parking\"][\"parking\"=\"surface\"]"
            f"({tx_miny},{tx_minx},{tx_maxy},{tx_maxx});"
            f"relation[\"amenity\"=\"parking\"][\"parking\"=\"surface\"]"
            f"({tx_miny},{tx_minx},{tx_maxy},{tx_maxx});"
            f"node[\"amenity\"=\"parking\"][\"parking\"=\"surface\"]"
            f"({tx_miny},{tx_minx},{tx_maxy},{tx_maxx}););out body geom;"
        )
        for ep in overpass_eps:
            for _ in range(3):
                try:
                    r = requests.get(ep, params={"data": query}, timeout=240)
                    r.raise_for_status()
                    return r.json()
                except Exception:
                    pass
        return {"elements": []}

    midx, midy = (minx + maxx) / 2, (miny + maxy) / 2
    tiles = [
        (minx, miny, midx, midy), (midx, miny, maxx, midy),
        (minx, midy, midx, maxy), (midx, midy, maxx, maxy),
    ]
    rows = []
    for tx_minx, tx_miny, tx_maxx, tx_maxy in tiles:
        for el in fetch_tile(tx_minx, tx_miny, tx_maxx, tx_maxy).get("elements", []):
            tags = el.get("tags", {})
            geom = None
            if "geometry" in el and len(el["geometry"]) >= 3:
                try:
                    geom = Polygon([(p["lon"], p["lat"]) for p in el["geometry"]])
                except Exception:
                    pass
            elif "lat" in el and "lon" in el:
                geom = Point(el["lon"], el["lat"])
            if geom:
                rows.append({
                    "osm_id": el.get("id"), "osm_type": el.get("type"),
                    "name": tags.get("name"), "geometry": geom,
                })
    if not rows:
        return gpd.GeoDataFrame(
            columns=["osm_id", "osm_type", "name", "geometry"],
            geometry="geometry", crs="EPSG:4326",
        )
    return (
        gpd.GeoDataFrame(rows, geometry="geometry", crs="EPSG:4326")
        .drop_duplicates(subset=["osm_type", "osm_id"])
        .reset_index(drop=True)
    )


def load_osm_surface_parking(bounds):
    if osm_cache.exists():
        print(f"Loading OSM parking cache: {osm_cache.name}")
        return gpd.read_parquet(osm_cache)
    osm_gdf = fetch_osm_surface_parking(bounds)
    osm_gdf.to_parquet(osm_cache, index=False)
    print(f"Saved: {osm_cache.name}  ({len(osm_gdf):,} lots)")
    return osm_gdf


## Step 1: Load and merge College Station parcel data

Run the diagnostic cell below first if you're seeing 0 PACS records or a GEO_ID error.

In [4]:
# Diagnostic cell — run once to verify PACS byte offsets and shapefile columns.
# Delete or skip after confirming offsets are correct.

# Delete bad caches so load functions re-download/re-parse from scratch.
for _bad in [geometry_cache, appraisal_cache]:
    if _bad.exists():
        _bad.unlink()
        print(f"Deleted bad cache: {_bad.name}")

# Show what PACS field extracts look like on the first raw line.
load_pacs_appraisal_info(diagnose=True)


Deleted bad cache: cs_geometry_2025cert.parquet
Deleted bad cache: cs_appraisal_info_2025cert.parquet
  Parsing: 2025-07-23_002022_APPRAISAL_INFO.TXT
  Line length: 9248 chars
  Key field extracts from first row:
    [0:12]  prop_id                = '000000010002'
    [12:17]  prop_type_cd           = 'R'
    [546:596]  geo_id                 = '000100-0001-0000'
    [608:678]  py_owner_name          = 'STASNY FAMILY RANCH LLC'
    [4459:4474]  situs_num              = ''
    [1049:1099]  situs_street           = 'SILVER HILL'
    [1109:1139]  situs_city             = ''
    [1139:1149]  situs_zip              = ''
    [1915:1930]  appraised_val          = '000000002865095'
    [1930:1945]  ten_percent_cap        = '000000000000000'
    [1945:1960]  assessed_val           = '000000002865095'
    [1795:1810]  land_hstd_val          = '000000000000000'
    [1810:1825]  land_non_hstd_val      = '000000000000000'
    [1825:1840]  imprv_hstd_val         = '000000000000000'
    [1840:1855]  

In [5]:
gdf_geom = load_shapefile()
df_prop  = load_pacs_appraisal_info()

# Normalise join key: shapefile GEO_ID ↔ PACS geo_id
gdf_geom["_geo_key"] = gdf_geom["GEO_ID"].astype(str).str.upper().str.strip()
df_prop["_geo_key"]  = df_prop["geo_id"].astype(str).str.upper().str.strip()

# Left join: keep all 28k CS shapefile parcels; PACS adds supplemental fields
gdf = gdf_geom.merge(df_prop, on="_geo_key", how="left")
gdf = gdf.drop(columns=["_geo_key"])

# Deduplicate: PACS can have multiple rows per geo_id (multi-owner UDI groups)
n_before = len(gdf)
gdf = gdf.drop_duplicates(subset=["GEO_ID"], keep="first")
if len(gdf) < n_before:
    print(f"  Deduplicated {n_before - len(gdf)} multi-owner PACS duplicates")

gdf = gpd.GeoDataFrame(gdf, geometry="geometry", crs="EPSG:4326")

pacs_matched = gdf["geo_id"].notna().sum() if "geo_id" in gdf.columns else gdf["prop_id"].notna().sum()
print(f"Shapefile parcels:    {len(gdf_geom):,}")
print(f"PACS records loaded:  {len(df_prop):,}")
print(f"Merged parcel count:  {len(gdf):,}")
print(f"PACS match rate:      {pacs_matched:,} / {len(gdf):,} ({pacs_matched/len(gdf)*100:.1f}%)")

# Numeric conversions for PACS fields (strings → numbers)
for col in [
    "appraised_val", "assessed_val", "ten_percent_cap",
    "land_hstd_val", "land_non_hstd_val",
    "imprv_hstd_val", "imprv_non_hstd_val",
]:
    if col in gdf.columns:
        gdf[col] = pd.to_numeric(gdf[col], errors="coerce").fillna(0)

# land_acres: prefer PACS (more precise) over shapefile column
pacs_acres = "land_acres_y" if "land_acres_y" in gdf.columns else "land_acres"
shp_acres  = "land_acres_x" if "land_acres_x" in gdf.columns else None
if shp_acres:
    gdf["land_acres"] = pd.to_numeric(gdf[pacs_acres], errors="coerce").fillna(
        pd.to_numeric(gdf[shp_acres], errors="coerce").fillna(0)
    )
else:
    gdf["land_acres"] = pd.to_numeric(gdf[pacs_acres], errors="coerce").fillna(0)

gdf["total_land_val"]  = gdf["land_hstd_val"]  + gdf["land_non_hstd_val"]
gdf["total_imprv_val"] = gdf["imprv_hstd_val"] + gdf["imprv_non_hstd_val"]

# Exemption flags: derive from shapefile Exemptions string.
# PACS flags (hs_exempt, ex_exempt) are all 'F' in the Brazos CAD 2025 export.
# EX-* codes (EX-XV etc.) = totally exempt state institution (TAMU, churches, govt).
ex_str = gdf["Exemptions"].fillna("").str.upper()
gdf["ex_exempt"]   = np.where(ex_str.str.contains(r"\bEX\b", regex=True), "T", "F")
gdf["hs_exempt"]   = np.where(ex_str.str.contains("HS"),   "T", "F")
gdf["ov65_exempt"] = np.where(ex_str.str.contains("OV65"), "T", "F")

# Owner name: prefer PACS py_owner_name, fall back to shapefile file_as_na
if "py_owner_name" not in gdf.columns or gdf["py_owner_name"].isna().all():
    gdf["py_owner_name"] = gdf.get("file_as_na", "")

# imprv_state_cd / land_state_cd: from PACS; fill missing with shapefile state_cd
if "imprv_state_cd" not in gdf.columns:
    gdf["imprv_state_cd"] = gdf.get("state_cd", "")
else:
    gdf["imprv_state_cd"] = gdf["imprv_state_cd"].fillna(gdf.get("state_cd", "")).fillna("")
if "land_state_cd" not in gdf.columns:
    gdf["land_state_cd"] = gdf.get("state_cd", "")
else:
    gdf["land_state_cd"] = gdf["land_state_cd"].fillna(gdf.get("state_cd", "")).fillna("")

# OSM surface-parking spatial join
osm_parking = load_osm_surface_parking(gdf.total_bounds)
if len(osm_parking) > 0:
    gdf_3857 = gdf.to_crs(epsg=3857)
    osm_3857 = osm_parking.to_crs(epsg=3857)
    joined   = gpd.sjoin(
        gdf_3857[["GEO_ID", "geometry"]],
        osm_3857[["osm_id", "geometry"]],
        how="left", predicate="intersects",
    )
    osm_geo_ids = set(joined.loc[joined["osm_id"].notna(), "GEO_ID"])
    gdf["is_osm_surface_parking"] = gdf["GEO_ID"].isin(osm_geo_ids)
else:
    gdf["is_osm_surface_parking"] = False

print(f"\nOSM surface-parking tagged: {int(gdf['is_osm_surface_parking'].sum()):,}")
print(f"ex_exempt='T': {(gdf['ex_exempt']=='T').sum():,}  (from Exemptions string, EX-* codes)")
print(f"hs_exempt='T': {(gdf['hs_exempt']=='T').sum():,}  (from Exemptions string)")
display(gdf[[
    "GEO_ID", "imprv_state_cd", "land_state_cd",
    "total_land_val", "total_imprv_val",
    "appraised_val", "assessed_val", "ex_exempt", "Exemptions",
]].head(5))


  Downloaded 0.0 MB
  Shapefile scores: [('Public_Parcel_Boundary_certified.shp', 10), ('Parcel_ID.shp', 2), ('Parcel_Land_Hook.shp', 2), ('Parcel_Xref.shp', 2), ('Street_Centerline.shp', 1)]
  Using: Public_Parcel_Boundary_certified_2025/Public_Parcel_Boundary_certified.shp
  Shapefile columns: ['PROP_ID', 'OBJECTID_1', 'prop_id_1', 'geo_id', 'Sale_Date', 'sl_price', 'sl_ratio', 'file_as_na', 'addr_line1', 'addr_line2', 'addr_line3', 'addr_city', 'addr_state', 'addr_zip', 'situs_num', 'situs_stre', 'situs_st_1', 'situs_st_2', 'situs_unit', 'legal_desc', 'abs_subdv_', 'Deed_Date', 'deed_book_', 'deed_book1', 'sl_type_cd', 'hood_cd', 'Entities', 'Exemptions', 'market', 'Land_Val', 'Imprv_Val', 'state_cd', 'ls_table', 'land_acres', 'land_sqft', 'living_are', 'class_cd', 'imprv_unit', 'yr_built', 'yr_blt', 'land_unit_', 'ImpCnt', 'Group_Code', 'Group_Co_1', 'Group_Co_2', 'Shape_Leng', 'Shape_Area', 'geometry']
  Total rows: 75,652
  Filtered on 'addr_city': 28,201 College Station parcels


,GEO_ID,imprv_state_cd,land_state_cd,total_land_val,total_imprv_val,appraised_val,assessed_val,ex_exempt,Exemptions
0,191000-0243-0100,A1,A1,33201.0,110696.0,143897.0,143897.0,F,None
1,191000-0257-0010,F1,F1,96775.0,902488.0,999263.0,999263.0,F,None
2,191500-0101-0160,O2,O2,26334.0,229754.0,256088.0,256088.0,F,None
3,191500-0202-0130,A7,A7,39750.0,198805.0,238555.0,238555.0,F,None
4,193100-0004-0150,F1,F1,507020.0,100.0,507120.0,507120.0,F,None


In [6]:
# Run this after Step 1 to deduplicate and inspect state codes
gdf = gdf.drop_duplicates(subset=["GEO_ID"], keep="first")
print(f"After dedup: {len(gdf):,} parcels")
print(gdf["imprv_state_cd"].value_counts().head(20).to_string())
print(f"ex_exempt value counts:\n{gdf['ex_exempt'].value_counts().to_string()}")


After dedup: 28,196 parcels
imprv_state_cd
A1     19936
        3378
F1      1046
A7       765
B2       628
E1       514
A8       464
A2       292
B4       215
A9       191
C1       156
D2       155
E4       130
EA2       81
O2        62
B1        55
A3        54
D1        32
F2        17
B3         8
ex_exempt value counts:
ex_exempt
F    27663
T      533


In [7]:
gdf["Exemptions"].value_counts().head(20)

Exemptions
HS                       9141
HS, OV65                 5544
EX-XV                     529
DV4, DVHS, HS             247
DV4, HS                   119
DV4, DVHS, HS, OV65       108
DP, HS                    107
DV1, HS, OV65              74
DV3, HS                    59
DV4, HS, OV65              55
DV2, HS                    46
DV2, HS, OV65              40
DV1, HS                    40
DV3, HS, OV65              39
DV4S, DVHSS, HS, OV65      30
DVHS, HS                   26
DV4                        19
DV4S, HS, OV65             15
DVHS, HS, OV65             11
HS, SO                      9
Name: count, dtype: int64

## Step 1b: Download zoning layer and stamp parcels

In [8]:
zoning_gdf = load_zoning_layer()

if len(zoning_gdf) > 0:
    gdf_3857    = gdf.to_crs(epsg=3857)
    zoning_3857 = zoning_gdf.to_crs(epsg=3857)

    gdf_3857["PARCEL_OID"] = gdf_3857["GEO_ID"]
    gdf_3857["centroid"]   = gdf_3857.geometry.centroid
    parcel_pts = gpd.GeoDataFrame(
        gdf_3857.drop(columns=["geometry"]), geometry="centroid", crs="EPSG:3857"
    )

    if "OBJECTID" in zoning_3857.columns:
        zoning_3857 = zoning_3857.rename(columns={"OBJECTID": "ZONING_OID"})
    else:
        zoning_3857["ZONING_OID"] = np.arange(len(zoning_3857), dtype=int)

    join_cols = [c for c in zoning_3857.columns if c != "geometry"]
    stamped   = gpd.sjoin(
        parcel_pts, zoning_3857[join_cols + ["geometry"]], how="left", predicate="within"
    )

    # Detect zoning-district text field
    zone_candidates = [
        "ZONING", "ZONE", "ZONE_NAME", "ZONETYPE", "ZONE_TYPE",
        "DISTRICT", "DISTRICTNAME", "LABEL", "NAME",
    ]
    chosen = next((c for c in zone_candidates if c in stamped.columns), None)
    if chosen is None:
        obj_cols = [c for c in join_cols if stamped[c].dtype == "object"]
        chosen   = obj_cols[0] if obj_cols else None

    if chosen:
        zone_map = stamped.groupby("PARCEL_OID")[chosen].first()
        gdf["ZONING_CLASS"] = gdf["GEO_ID"].map(zone_map)
        print(f"Zoning field used: '{chosen}'")
    else:
        gdf["ZONING_CLASS"] = None
        print("Warning: no zoning text field detected; ZONING_CLASS left null.")
else:
    gdf["ZONING_CLASS"] = None

print(f"Parcels stamped with zoning: {int(gdf['ZONING_CLASS'].notna().sum()):,}")
display(gdf[["GEO_ID", "imprv_state_cd", "ZONING_CLASS"]].head(10))


Primary zoning endpoint failed (HTTPSConnectionPool(host='maps.cstx.gov', port=443): Max retries exceeded with url: /cstx/rest/services/OpenData_PDS/FeatureServer/14/query?f=json&where=1%3D1&returnCountOnly=true (Caused by ConnectTimeoutError(<HTTPSConnection(host='maps.cstx.gov', port=443) at 0x1cdbf369bd0>, 'Connection to maps.cstx.gov timed out. (connect timeout=60)'))); trying open-data portal.
Parcels stamped with zoning: 0


,GEO_ID,imprv_state_cd,ZONING_CLASS
0,191000-0243-0100,A1,None
1,191000-0257-0010,F1,None
2,191500-0101-0160,O2,None
3,191500-0202-0130,A7,None
4,193100-0004-0150,F1,None
5,191000-0180-0030,A1,None
6,191000-0230-0010,A2,None
7,194000-0110-0030,A1,None
8,195400-1020-0010,,None
9,195400-0030-0050,A1,None


## Step 2: Quality checks and entity-specific current tax

In [9]:
# Valuation consistency
gdf["val_check"] = gdf["total_land_val"] + gdf["total_imprv_val"]
gdf["val_diff"]  = (gdf["appraised_val"] - gdf["val_check"]).abs()

taxable = gdf[gdf["appraised_val"] > 0]
print(f"Parcels with appraised_val > 0: {len(taxable):,}")
print(f"Max land+imprv vs appraised mismatch: ${taxable['val_diff'].max():,.0f}")
print(f"Median mismatch:                      ${taxable['val_diff'].median():,.0f}")
# Note: large mismatches occur on ag-deferred or timber parcels — expected.

n_exempt = (gdf["ex_exempt"] == "T").sum()
print(f"\nFully exempt (TAMU/state/church, ex_exempt='T'): {n_exempt:,} of {len(gdf):,}")

# Improvement ratio
gdf["IR"] = np.where(
    gdf["appraised_val"] > 0,
    gdf["total_imprv_val"] / gdf["appraised_val"],
    0,
)

# Entity-specific current tax
# City of College Station: 20% homestead + $30k over-65
cs_hs = np.where(gdf["hs_exempt"] == "T", CS_HOMESTEAD_RATE * gdf["appraised_val"], 0)
cs_o65 = np.where(gdf["ov65_exempt"] == "T", CS_OV65_AMT, 0)
gdf["cs_taxable"] = np.where(
    gdf["ex_exempt"] == "T", 0,
    np.maximum(0, gdf["assessed_val"] - cs_hs - cs_o65),
)

# Brazos County: update BC_HOMESTEAD_AMT / BC_OV65_AMT if county offers optional exemptions
bc_hs  = np.where(gdf["hs_exempt"] == "T", BC_HOMESTEAD_AMT, 0)
bc_o65 = np.where(gdf["ov65_exempt"] == "T", BC_OV65_AMT, 0)
gdf["bc_taxable"] = np.where(
    gdf["ex_exempt"] == "T", 0,
    np.maximum(0, gdf["assessed_val"] - bc_hs - bc_o65),
)

# College Station ISD: $40k mandatory homestead + $10k additional over-65
csisd_hs  = np.where(gdf["hs_exempt"] == "T", CSISD_HOMESTEAD_AMT, 0)
csisd_o65 = np.where(gdf["ov65_exempt"] == "T", CSISD_OV65_AMT, 0)
gdf["csisd_taxable"] = np.where(
    gdf["ex_exempt"] == "T", 0,
    np.maximum(0, gdf["assessed_val"] - csisd_hs - csisd_o65),
)

gdf["current_tax_cs"]    = gdf["cs_taxable"]    * MILLAGE_CS    / 100
gdf["current_tax_bc"]    = gdf["bc_taxable"]    * MILLAGE_BC    / 100
gdf["current_tax_csisd"] = gdf["csisd_taxable"] * MILLAGE_CSISD / 100
gdf["current_tax"]       = gdf["current_tax_cs"] + gdf["current_tax_bc"] + gdf["current_tax_csisd"]

print("\nCombined levy modeled (CS + BC + CSISD):")
print(f"  Total:  ${gdf['current_tax'].sum():>15,.0f}")
print(f"  CS:     ${gdf['current_tax_cs'].sum():>15,.0f}")
print(f"  BC:     ${gdf['current_tax_bc'].sum():>15,.0f}")
print(f"  CSISD:  ${gdf['current_tax_csisd'].sum():>15,.0f}")

display(gdf[["assessed_val", "cs_taxable", "bc_taxable", "csisd_taxable", "current_tax"]].describe())


Parcels with appraised_val > 0: 27,870
Max land+imprv vs appraised mismatch: $35,957,495
Median mismatch:                      $0

Fully exempt (TAMU/state/church, ex_exempt='T'): 533 of 28,196

Combined levy modeled (CS + BC + CSISD):
  Total:  $    223,824,289
  CS:     $     55,178,734
  BC:     $     52,763,968
  CSISD:  $    115,881,587


,assessed_val,cs_taxable,bc_taxable,csisd_taxable,current_tax
count,2.819600e+04,2.819600e+04,2.819600e+04,2.819600e+04,28196.000000
mean,5.072085e+05,3.823163e+05,4.458728e+05,4.213943e+05,7938.157495
std,1.641093e+06,7.345471e+05,7.479641e+05,7.466628e+05,14171.059648
min,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000
25%,2.583828e+05,2.125744e+05,2.529768e+05,2.289412e+05,4409.121437
50%,3.525270e+05,2.943143e+05,3.488720e+05,3.205000e+05,6101.036964
75%,5.233235e+05,4.250010e+05,5.132778e+05,4.810020e+05,9032.331100
max,1.253575e+08,4.200000e+07,4.200000e+07,4.200000e+07,800886.240000


In [10]:
gdf["ZONING_CLASS"].value_counts()

Series([], Name: count, dtype: int64)

## Step 3: Property categories for summary output

In [11]:
def map_property_category(row) -> str:
    ex  = str(row.get("ex_exempt",      "")).strip().upper()
    isc = str(row.get("imprv_state_cd", "")).strip().upper()
    lsc = str(row.get("land_state_cd",  "")).strip().upper()

    if ex == "T":
        return "Exempt / Government"

    for code in [isc, lsc]:
        if not code:
            continue
        if code.startswith("A"):              # A1=SF, A2=mobile home, A7/A8/A9=Brazos CAD variants
            return "Residential"
        if code in ("B1", "B2", "B3", "B4"):  # multi-family
            return "Residential"
        if code in ("C1", "C2"):              # vacant lot / commercial vacant
            return "Vacant / Undeveloped"
        if code.startswith("D"):              # ag / open space
            return "Agricultural"
        if code == "F2":
            return "Industrial"
        if code.startswith("F"):              # F1 commercial real
            return "Commercial"
        if code.startswith(("G", "J", "X", "E")):  # gov / utility / exempt / Brazos CAD E-prefix exempt
            return "Exempt / Government"
        if code.startswith("O"):              # residential inventory (O2)
            return "Residential"
    return "Other"


def map_refined_category(base: str, imprv_state_cd: str, zoning_class: str) -> str:
    isc = str(imprv_state_cd).strip().upper()
    # Treat None / "nan" / "none" as missing zoning
    z_raw = str(zoning_class).strip().upper()
    z = "" if z_raw in ("", "NONE", "NAN", "NULL") else z_raw

    if base in ("Exempt / Government", "Vacant / Undeveloped", "Agricultural",
                "Commercial", "Industrial"):
        return base

    if base == "Residential":
        if isc == "B1":
            # Student housing: large apartments in MF / T / PDD zones near TAMU
            if z and (any(tok in z for tok in ["MF", "PDD", "P-MUD"]) or z in ("T", "MF")):
                return "Residential - Student Housing"
            return "Residential - High Density"
        if isc in ("B2", "B3", "B4"):
            return "Residential - Medium Density"
        # A* codes = single-family per Texas Comptroller classification system.
        # Use zoning to confirm when available; fall back to state code.
        if z and (any(tok in z for tok in ["GS", "RS", "R-1", "R1"]) or z.startswith("R-") or z == "R"):
            return "Residential - Low Density"
        if isc.startswith("A") or isc in ("O2", "A3"):
            # A1, A2, A3, A7, A8, A9 are all single-family variants
            return "Residential - Low Density"
        return "Residential - Other"

    if z and any(tok in z for tok in ["MH", "MU", "WPC"]):
        return "Mixed Use"

    return base


gdf["PROPERTY_CATEGORY"] = gdf.apply(map_property_category, axis=1)
gdf["ANALYSIS_CATEGORY"] = gdf.apply(
    lambda r: map_refined_category(
        r["PROPERTY_CATEGORY"], r.get("imprv_state_cd", ""), r.get("ZONING_CLASS", "")
    ),
    axis=1,
)

display(gdf["ANALYSIS_CATEGORY"].value_counts().to_frame("parcel_count"))

analysis_gdf = gdf[
    gdf["PROPERTY_CATEGORY"].ne("Exempt / Government")
    & gdf["PROPERTY_CATEGORY"].ne("Agricultural")
].copy()
print(f"
Excluded exempt/ag: {len(gdf) - len(analysis_gdf):,}")
print(f"Parcels for LVT modeling: {len(analysis_gdf):,}")

SyntaxError: unterminated string literal (detected at line 78) (935005695.py, line 78)

## Step 3b: Export mapping-ready GeoParquet

In [ ]:
export_cols = [
    "GEO_ID", "prop_id", "py_owner_name",
    "imprv_state_cd", "land_state_cd",
    "PROPERTY_CATEGORY", "ANALYSIS_CATEGORY", "ZONING_CLASS",
    "is_osm_surface_parking",
    "land_acres", "total_land_val", "total_imprv_val",
    "appraised_val", "assessed_val",
    "current_tax",
    "geometry",
]
export_gdf = analysis_gdf[[c for c in export_cols if c in analysis_gdf.columns]].copy()
export_gdf = export_gdf.to_crs(epsg=4326)
export_gdf.to_parquet(mapping_cache, index=False)
print(f"Wrote mapping GeoParquet: {mapping_cache}")
print(f"CRS: {export_gdf.crs}  |  Rows: {len(export_gdf):,}")
display(export_gdf.head(5))


## Step 4: Revenue-neutral split-rate scenarios

In [ ]:
model_df        = analysis_gdf.copy()
current_revenue = model_df["current_tax"].sum()
print(f"Current combined levy (CS + BC + CSISD): ${current_revenue:,.0f}")

scenario_outputs = {}
for ratio in [2, 4, 8]:
    land_mill, imp_mill, revenue, scenario_df = model_split_rate_tax(
        df=model_df.copy(),
        land_value_col="total_land_val",
        improvement_value_col="total_imprv_val",
        current_revenue=current_revenue,
        land_improvement_ratio=ratio,
    )
    new_col = f"tax_shift_{ratio}to1"
    scenario_df[new_col] = scenario_df["new_tax"] - scenario_df["current_tax"]
    scenario_outputs[ratio] = {
        "land_mill": land_mill, "imp_mill": imp_mill,
        "revenue": revenue, "df": scenario_df, "new_col": new_col,
    }
    print(f"{ratio}:1  land mill={land_mill:.4f}  imp mill={imp_mill:.4f}  revenue=${revenue:,.0f}")

# Universal Building Exemption: land is the sole tax base; no homestead or other exemptions.
# Revenue-neutral rate = current_revenue / total_land_value.
ube_land_total = model_df["total_land_val"].sum()
ube_rate       = (current_revenue * 1000) / ube_land_total  # mills (per ,000)
ube_df         = model_df.copy()
ube_df["new_tax"]      = ube_df["total_land_val"] * ube_rate / 1000
ube_df["tax_shift_ube"] = ube_df["new_tax"] - ube_df["current_tax"]
scenario_outputs["UBE"] = {
    "land_mill": ube_rate, "imp_mill": 0.0,
    "revenue": ube_df["new_tax"].sum(), "df": ube_df, "new_col": "tax_shift_ube",
}
print(f"UBE  land mill={ube_rate:.4f}  imp mill=0.0000  revenue=${ube_df['new_tax'].sum():,.0f}")
print(f"     (land-only tax base: ${ube_land_total:,.0f})")

In [ ]:
# ── Scenario selector ────────────────────────────────────────────────────────
# Set DISPLAY_SCENARIO to one of: "2:1", "4:1", "8:1", "UBE"
DISPLAY_SCENARIO = "4:1"

if DISPLAY_SCENARIO == "UBE":
    primary        = scenario_outputs["UBE"]["df"].copy()
    scenario_label = "Universal Building Exemption (land-only tax)"
else:
    _ratio         = int(DISPLAY_SCENARIO.split(":")[0])
    primary        = scenario_outputs[_ratio]["df"].copy()
    scenario_label = f"{DISPLAY_SCENARIO} split-rate LVT"

primary["tax_change"]     = primary["new_tax"] - primary["current_tax"]
primary["tax_change_pct"] = np.where(
    primary["current_tax"] > 0,
    primary["tax_change"] / primary["current_tax"] * 100,
    0,
)
print(f"Active scenario: {scenario_label}")
print(f"Revenue: ${primary['new_tax'].sum():,.0f}  (current: ${primary['current_tax'].sum():,.0f})")

## Step 5: Tax-capacity split model and underdevelopment analysis

In [ ]:
cs_city = analysis_gdf.copy()
cs_city["TaxCapacity"]              = cs_city["assessed_val"]
cs_city["TaxCapacity_Improvements"] = cs_city["IR"] * cs_city["TaxCapacity"]
cs_city["TaxCapacity_Land"]         = (1 - cs_city["IR"]) * cs_city["TaxCapacity"]
current_revenue_tc                  = cs_city["current_tax"].sum()

print("Tax Capacity Split Summary:")
total_tc = cs_city["TaxCapacity"].sum()
print(f"  Total Tax Capacity:          ${total_tc:>15,.0f}")
print(f"  Tax Capacity (Improvements): ${cs_city['TaxCapacity_Improvements'].sum():>15,.0f}")
print(f"  Tax Capacity (Land):         ${cs_city['TaxCapacity_Land'].sum():>15,.0f}")
print(f"  Land % of Tax Capacity:      {cs_city['TaxCapacity_Land'].sum() / total_tc * 100:.1f}%")

tc_ratio = 2
tc_land_mill, tc_imp_mill, tc_revenue, cs_city = model_split_rate_tax(
    df=cs_city,
    land_value_col="TaxCapacity_Land",
    improvement_value_col="TaxCapacity_Improvements",
    current_revenue=current_revenue_tc,
    land_improvement_ratio=tc_ratio,
)
cs_city["new_tax_tc"]        = cs_city["new_tax"]
cs_city["tax_change_tc"]     = cs_city["new_tax_tc"] - cs_city["current_tax"]
cs_city["tax_change_pct_tc"] = np.where(
    cs_city["current_tax"] > 0,
    cs_city["tax_change_tc"] / cs_city["current_tax"] * 100,
    0,
)

print(f"\nTC Split-Rate ({tc_ratio}:1)")
print(f"  Land millage:    {tc_land_mill:.6f}")
print(f"  Imp millage:     {tc_imp_mill:.6f}")
print(f"  Revenue neutral: {abs(current_revenue_tc - cs_city['new_tax_tc'].sum()) < 1}")

# Underdeveloped land analysis
cs_city["LAND_DEV_CATEGORY"] = np.where(cs_city["IR"] == 0, "Vacant Land", "Developed")
vacant_results = analyze_vacant_land(
    df=cs_city,
    land_value_col="total_land_val",
    improvement_value_col="total_imprv_val",
    property_type_col="LAND_DEV_CATEGORY",
    vacant_identifier="Vacant Land",
)
underdeveloped = analyze_land_by_improvement_share(
    df=cs_city,
    land_value_col="total_land_val",
    improvement_value_col="total_imprv_val",
)
total_land_value = cs_city["total_land_val"].sum()

print(f"\nUndeveloped and Underdeveloped Land")
print(f"  Total non-exempt land value: ${total_land_value:,.0f}")
if "error" not in vacant_results:
    print(
        f"  Vacant (IR=0): {vacant_results['total_vacant_parcels']:,} parcels, "
        f"${vacant_results['total_vacant_land_value']:,.0f} "
        f"({vacant_results['vacant_land_pct_of_total']:.1f}% of non-exempt land value)"
    )
print("\n  By improvement share:")
for row in underdeveloped["categories"]:
    print(
        f"    {row['category']:<35} {row['parcel_count']:>6,} parcels  "
        f"${row['adjusted_land_value']:>14,.0f}  ({row['share_of_total_land_value_pct']:.1f}%)"
    )


## Step 6: Category summary & charts

In [ ]:
output_summary = calculate_category_tax_summary(
    df=primary,
    category_col="ANALYSIS_CATEGORY",
    current_tax_col="current_tax",
    new_tax_col="new_tax",
)
print_category_tax_summary(output_summary)

plot_df = (
    output_summary[output_summary["property_count"] > 50]
    .sort_values("median_tax_change_pct")
    .copy()
)
fig_height = max(5, len(plot_df) * 0.7)
plt.figure(figsize=(12, fig_height))
bar_colors = np.where(plot_df["median_tax_change_pct"] < 0, "#228B22", "#8B0000")
plt.barh(plot_df["ANALYSIS_CATEGORY"], plot_df["median_tax_change_pct"], color=bar_colors)
plt.axvline(0, color="black", linewidth=1)
plt.title(f"Median tax change % by property category ({scenario_label})")
plt.xlabel("Median tax change (%)")
plt.tight_layout()
plt.show()


In [ ]:
summary_filtered = output_summary[output_summary["property_count"] > 50].copy()
summary_sorted   = summary_filtered.sort_values("pct_increase_gt_threshold", ascending=True)

cats      = summary_sorted["ANALYSIS_CATEGORY"].tolist()
pct_inc   = summary_sorted["pct_increase_gt_threshold"].tolist()
pct_dec   = summary_sorted["pct_decrease_gt_threshold"].tolist()
pct_inc_i = [int(round(x)) for x in pct_inc]
pct_dec_i = [int(round(x)) for x in pct_dec]

y = np.arange(len(cats))
fig, ax = plt.subplots(figsize=(8, max(5, len(cats) * 0.6)))

color_inc = "#8B0000"
color_dec = "#228B22"

ax.barh(y, [-v for v in pct_dec], color=color_dec, edgecolor="none", height=0.7)
ax.barh(y, pct_inc, color=color_inc, edgecolor="none", height=0.7)

for i, (inc, dec) in enumerate(zip(pct_inc_i, pct_dec_i)):
    if dec > 0:
        ax.text(-dec - 2, y[i], f"{dec}%", va="center", ha="right", fontsize=8, color=color_dec)
    if inc > 0:
        ax.text(inc + 2, y[i], f"{inc}%", va="center", ha="left",  fontsize=8, color=color_inc)

max_val = max(max(pct_inc), max(pct_dec)) if pct_inc else 10
for i, (cat, inc) in enumerate(zip(cats, pct_inc)):
    ax.text(
        inc + 18 if inc > 0 else 18, y[i], cat,
        va="center", ha="left", fontsize=9, fontweight="bold", color="#222",
    )

for spine in ax.spines.values():
    spine.set_visible(False)
ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
ax.set_yticks([])
ax.set_xticks([])
ax.set_xlim(-max_val - 20, max_val + 50)

title_y = len(cats) - 0.2
ax.text(-max_val * 0.45, title_y, "Percent of parcels\ndecreasing >10%",
        ha="center", va="bottom", fontsize=10)
ax.text( max_val * 0.45, title_y, "Percent of parcels\nincreasing >10%",
        ha="center", va="bottom", fontsize=10)

plt.tight_layout()
plt.show()


## Step 7: Land-use diagnostics (vacancy, parking, low-improvement share)

In [ ]:
analysis_df = primary.copy()
analysis_df["LAND_USE_FOR_ANALYSIS"] = np.select(
    [
        analysis_df["PROPERTY_CATEGORY"].str.contains("Vacant", na=False),
        analysis_df["is_osm_surface_parking"] == True,
    ],
    ["Vacant Land", "Trans - Parking"],
    default="Other",
)

vacant_results = analyze_vacant_land(
    df=analysis_df,
    land_value_col="total_land_val",
    improvement_value_col="total_imprv_val",
    property_type_col="LAND_USE_FOR_ANALYSIS",
    vacant_identifier="Vacant Land",
    owner_col="py_owner_name",
)
print_vacant_land_summary(vacant_results)

parking_results = analyze_parking_lots(
    df=analysis_df,
    land_value_col="total_land_val",
    improvement_value_col="total_imprv_val",
    property_type_col="LAND_USE_FOR_ANALYSIS",
    parking_identifier="Trans - Parking",
    min_land_value_threshold=50_000,
    max_improvement_ratio=0.10,
)
print_parking_analysis_summary(parking_results)

low_impr_share = analyze_land_by_improvement_share(
    df=analysis_df,
    land_value_col="total_land_val",
    improvement_value_col="total_imprv_val",
)
display(pd.DataFrame(low_impr_share["categories"]))


## Step 7b: High-impact parcels

In [ ]:
show_cols = [
    "GEO_ID", "py_owner_name", "imprv_state_cd", "ANALYSIS_CATEGORY",
    "total_land_val", "total_imprv_val",
    "current_tax", "new_tax", "tax_change", "tax_change_pct",
]
available = [c for c in show_cols if c in primary.columns]
top_increase = primary.nlargest(20,  "tax_change")[available]
top_decrease = primary.nsmallest(20, "tax_change")[available]

print("Top 20 increases")
display(top_increase)
print("Top 20 decreases")
display(top_decrease)


## Step 8: Census equity analysis

In [ ]:
equity_df = primary.copy().to_crs(epsg=4326)

try:
    census_data, census_boundaries = get_census_data_with_boundaries(
        fips_code=BRAZOS_COUNTY_FIPS,
        year=2022,
    )
    matched = match_to_census_blockgroups(equity_df, census_boundaries, join_type="left")
    matched = matched[matched["median_income"] > 0].copy()

    bg_summary = (
        matched.groupby("std_geoid")
        .agg(
            median_income       =("median_income",    "first"),
            minority_pct        =("minority_pct",     "first"),
            total_current_tax   =("current_tax",      "sum"),
            total_new_tax       =("new_tax",           "sum"),
            mean_tax_change     =("tax_change",        "mean"),
            median_tax_change   =("tax_change",        "median"),
            median_tax_change_pct=("tax_change_pct",  "median"),
            parcel_count        =("GEO_ID",            "count"),
            has_vacant_land     =("ANALYSIS_CATEGORY", lambda x: (x == "Vacant / Undeveloped").any()),
        )
        .reset_index()
    )
    bg_summary = bg_summary[bg_summary["median_income"] > 0].copy()
    bg_summary["mean_tax_change_pct"] = np.where(
        bg_summary["total_current_tax"] > 0,
        (bg_summary["total_new_tax"] - bg_summary["total_current_tax"])
        / bg_summary["total_current_tax"] * 100,
        0,
    )
    non_vacant_bg = bg_summary[~bg_summary["has_vacant_land"]].copy()

    income_q    = create_quintile_summary(bg_summary,    "median_income", "median_income", "mean_tax_change", "mean_tax_change_pct")
    nv_income_q = create_quintile_summary(non_vacant_bg, "median_income", "median_income", "mean_tax_change", "mean_tax_change_pct")
    minority_q  = create_quintile_summary(bg_summary,    "minority_pct",  "minority_pct",  "mean_tax_change", "mean_tax_change_pct")
    nv_minor_q  = create_quintile_summary(non_vacant_bg, "minority_pct",  "minority_pct",  "mean_tax_change", "mean_tax_change_pct")

    display(income_q)
    display(nv_income_q)
    display(minority_q)
    display(nv_minor_q)

    # Line charts
    for q_all, q_nv, x_col, title in [
        (income_q,   nv_income_q, "median_income_quintile",
         "Mean tax change % by neighborhood income quintile"),
        (minority_q, nv_minor_q,  "minority_pct_quintile",
         "Mean tax change % by minority-share quintile"),
    ]:
        plt.figure(figsize=(10, 5))
        plt.plot(q_all[x_col], q_all["mean_tax_change_pct"], marker="o", label="All properties")
        plt.plot(q_nv[x_col],  q_nv["mean_tax_change_pct"],  marker="o",
                 label="Excl. vacant-land neighborhoods")
        plt.axhline(0, color="black", linewidth=1, linestyle="dotted")
        plt.title(title)
        plt.xlabel(x_col.replace("_", " ").title())
        plt.ylabel("Mean tax change (%)")
        plt.legend()
        plt.tight_layout()
        plt.show()

    # Inverted bar charts (median, excl. vacant)
    sns.set_theme(style="whitegrid", font_scale=1.15)

    def inverted_bar(quintile_df, lbl_col, palette, title):
        fig, ax = plt.subplots(figsize=(10, 6))
        vals      = quintile_df["median_tax_change_pct"]
        labels    = quintile_df[lbl_col]
        colors    = sns.color_palette(palette, n_colors=len(vals))
        color_map = [colors[i] for i in np.argsort(np.argsort(-vals))]
        bars      = ax.bar(labels, vals, color=color_map, edgecolor="black", width=0.7)
        ax.yaxis.set_visible(False)
        ax.set_title(title, weight="bold", pad=30)
        sns.despine(left=True, right=True, top=True, bottom=True)
        for bar, val in zip(bars, vals):
            ax.annotate(
                f"{val:.1f}%",
                xy=(bar.get_x() + bar.get_width() / 2, val / 2),
                ha="center", va="center", fontsize=13, fontweight="bold",
            )
        ax.xaxis.set_ticks_position("top")
        ax.xaxis.set_label_position("top")
        plt.xticks(fontweight="bold")
        margin = max(abs(vals.min()), abs(vals.max())) * 1.2 if len(vals) else 1
        ax.set_ylim(-margin, margin)
        ax.axhline(y=0, color="black", linewidth=0.8)
        plt.tight_layout()
        plt.show()

    inverted_bar(nv_income_q, "median_income_quintile", "Greens",
                 "Median Tax Change by Neighborhood Income (Excl. Vacant)")
    inverted_bar(nv_minor_q,  "minority_pct_quintile",  "Purples",
                 "Median Tax Change by Neighborhood Minority % (Excl. Vacant)")

    # Subgroup quintile charts
    def render_quintile_bars(df_subset, title_prefix):
        if len(df_subset) < 25:
            print(f"Skipping {title_prefix}: too few parcels ({len(df_subset)}).")
            return
        inc   = create_quintile_summary(df_subset, "median_income", "median_income",
                                        "tax_change", "tax_change_pct")
        minor = create_quintile_summary(df_subset, "minority_pct",  "minority_pct",
                                        "tax_change", "tax_change_pct")
        for summ, lbl_col, palette, ttl in [
            (inc,   "median_income_quintile", "Greens",
             f"Median Tax Change by Income ({title_prefix})"),
            (minor, "minority_pct_quintile",  "Purples",
             f"Median Tax Change by Minority % ({title_prefix})"),
        ]:
            inverted_bar(summ, lbl_col, palette, ttl)

    render_quintile_bars(
        matched[matched["ANALYSIS_CATEGORY"].str.contains("Low Density",    na=False)],
        "Residential - Low Density",
    )
    render_quintile_bars(
        matched[matched["ANALYSIS_CATEGORY"].str.contains("Medium Density", na=False)],
        "Residential - Medium Density",
    )
    render_quintile_bars(
        matched[matched["ANALYSIS_CATEGORY"].str.contains("High Density",   na=False)],
        "Residential - High Density",
    )
    render_quintile_bars(
        matched[matched["ANALYSIS_CATEGORY"].str.contains("Student",        na=False)],
        "Residential - Student Housing",
    )
    render_quintile_bars(
        matched[matched["PROPERTY_CATEGORY"].str.contains("Residential",    na=False)],
        "All Residential",
    )

except Exception as exc:
    print("Census equity analysis skipped:", exc)
    print("Set CENSUS_API_KEY in .env and rerun this cell.")
